# 04 - Visualization
**Social Media Mood Analyzer**  
This notebook generates all evaluation visualizations:
- Confusion Matrix (raw counts and normalised)
- Per-class Precision, Recall, and F1-Score bar chart

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

MOOD_LABELS = ['happy', 'sad', 'angry', 'neutral']
os.makedirs('results/charts', exist_ok=True)

print('Libraries imported successfully.')

## 2. Load Data, Model and Generate Predictions

In [ ]:
df = pd.read_csv('data/processed/mood_data.csv')

X = df['cleaned_text']
y = df['mood']

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model      = joblib.load('results/model.pkl')
vectorizer = joblib.load('results/vectorizer.pkl')

y_pred = model.predict(vectorizer.transform(X_test))

print(f'Predictions ready: {len(y_pred)} samples')

## 3. Confusion Matrix (Raw Counts)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=MOOD_LABELS)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=MOOD_LABELS, yticklabels=MOOD_LABELS,
            linewidths=0.5, ax=ax)
ax.set_title('Confusion Matrix (Raw Counts)', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Mood', fontsize=11)
ax.set_ylabel('True Mood', fontsize=11)
plt.tight_layout()
plt.savefig('results/charts/confusion_matrix_raw.png', dpi=150)
plt.show()
print('Saved: results/charts/confusion_matrix_raw.png')

## 4. Confusion Matrix (Normalised)

In [ ]:
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=MOOD_LABELS, yticklabels=MOOD_LABELS,
            linewidths=0.5, ax=ax)
ax.set_title('Confusion Matrix (Normalised)', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Mood', fontsize=11)
ax.set_ylabel('True Mood', fontsize=11)
plt.tight_layout()
plt.savefig('results/charts/confusion_matrix_normalised.png', dpi=150)
plt.show()
print('Saved: results/charts/confusion_matrix_normalised.png')

## 5. Per-Class Metrics Bar Chart

In [ ]:
report = classification_report(
    y_test, y_pred,
    target_names=MOOD_LABELS,
    output_dict=True,
    zero_division=0
)

precision = [report[c]['precision'] for c in MOOD_LABELS]
recall    = [report[c]['recall']    for c in MOOD_LABELS]
f1        = [report[c]['f1-score']  for c in MOOD_LABELS]

x     = np.arange(len(MOOD_LABELS))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width, precision, width, label='Precision', color='#4c72b0')
ax.bar(x,         recall,    width, label='Recall',    color='#55a868')
ax.bar(x + width, f1,        width, label='F1-Score',  color='#c44e52')

ax.set_title('Per-Class Evaluation Metrics', fontsize=14, fontweight='bold')
ax.set_xlabel('Mood Category', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(MOOD_LABELS, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('results/charts/per_class_metrics.png', dpi=150)
plt.show()
print('Saved: results/charts/per_class_metrics.png')

## 6. Summary
All charts have been saved to `results/charts/`:
- `confusion_matrix_raw.png`
- `confusion_matrix_normalised.png`
- `per_class_metrics.png`